# 0.5 · Matplotlib & Seaborn 静态可视化

> **课程定位 / Where this fits**
> 第 5 课，**Part 0 · 基础准备**。
> Lesson 5, **Part 0 · Foundations**.
>
> Pandas 让你"看到数字"，Matplotlib + Seaborn 让你"看到图形"。**任何 EDA 报告、模型评估、面试演示都要靠它**。
> Pandas shows you the numbers; Matplotlib + Seaborn show you the picture. **Every EDA report, model eval, interview demo depends on it.**

> 📐 **符号约定 / Notation**（见 [`NOTATION.md`](../NOTATION.md)）
> 本节几乎都是 API 使用，不涉及推导。涉及统计时遵循 $n$ = 样本数、$\bar{x}$ = 样本均值、$s$ = 样本标准差。
> Mostly API usage. When stats appear: $n$ samples, $\bar{x}$ mean, $s$ std.

> 💡 **面试相关 / Interview-relevant**
> 数据可视化在 DS 面试里属于"**讲数据故事**"的能力评估。常见考点：
> - 给你一个图，让你解读 / 找异常
> - 给你一个分析问题，让你说应该画什么图
> - presentation 阶段："你怎么向不懂技术的 stakeholder 讲清这个结论"
>
> Interviewers test storytelling: read a plot, pick the right plot, explain to non-technical stakeholders.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 解释 Matplotlib 的 **Figure / Axes 对象模型**——为什么 `fig, ax = plt.subplots()` 是工业标准写法。
   Explain the **Figure / Axes** object model — why `fig, ax = plt.subplots()` is the standard pattern.
2. 区分 **stateful API (`plt.xxx`)** 和 **OO API (`ax.xxx`)** 的使用场景。
   Tell apart stateful (`plt.xxx`) vs OO (`ax.xxx`) APIs and when to use each.
3. 熟练画 15+ 种核心图表：line / scatter / bar / hist / box / violin / heatmap / pairplot / facet ...
4. 用 Seaborn 一行画**带统计意义的图**（分布 + KDE / 回归线 + CI）。
   Use seaborn to render statistically meaningful plots (distributions + KDE, regression + CI) in one line.
5. 调出**可发表/可演示级别**的图：字体、配色、注释、保存为高分辨率。
   Produce publication-grade plots: fonts, palettes, annotations, high-DPI export.
6. 在 **Tips** 数据集上完成完整 EDA 可视化。
   Deliver an end-to-end EDA on the **Tips** dataset.

---

## 目录 / Table of Contents

1. [🍽 Tips 数据集介绍 / Dataset Intro](#1)
2. [Matplotlib 对象模型 / The Object Model](#2)
3. [Stateful API vs OO API](#3)
4. [基础图表 / Core Plot Types](#4)
5. [Subplot 布局 / Subplot Layouts](#5)
6. [风格定制 / Styling](#6)
7. [注释 / Annotations](#7)
8. [Seaborn 简介 / Seaborn Intro](#8)
9. [分布图 / Distribution Plots](#9)
10. [类别图 / Categorical Plots](#10)
11. [关系图 / Relational Plots](#11)
12. [矩阵图 / Heatmap & Clustermap](#12)
13. [FacetGrid & pairplot](#13)
14. [保存图片 / Saving Figures](#14)
15. [实战：Tips 完整 EDA / Hands-on](#15)
16. [小结 / Summary](#16)


<a id="1"></a>
## 1. 🍽 Tips 数据集介绍 / Dataset Intro

> **来源 / Source**: Bryant & Smith (1995) *Practical Data Analysis*; seaborn 内置。
> Bryant & Smith (1995); built into seaborn.
>
> **背景 / Background**: 美国某餐厅服务员收集的 **244 顿饭**记录，主要研究"小费比例受什么影响"。
> 244 dinners recorded by a US waiter; studies what drives tip ratios.
>
> | 列 / Column | 含义 / Meaning | 类型 |
> |---|---|---|
> | `total_bill` | 总账单 ($) / total bill | float |
> | `tip`        | 小费 ($) / tip | float |
> | `sex`        | 顾客性别 / payer's sex | category |
> | `smoker`     | 是否吸烟 / smoker | category |
> | `day`        | 星期几 / day of week (Thu–Sun) | category |
> | `time`       | 午餐/晚餐 / Lunch or Dinner | category |
> | `size`       | 同桌人数 / party size | int |
>
> **任务 / Task**: 探索小费规律（本节只做可视化，不建模）。
> Explore tipping patterns; viz only, no modeling.
>
> **为什么经典 / Why classic**: 小、干净、有数值 + 多种类别 → **可视化教学的标准集**。
> Small, clean, numeric + multiple categoricals — the textbook viz dataset.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 让图在 notebook 中内联显示 / Inline display in notebook
# 注意：matplotlib 3.9+ 默认就是 inline，不再需要 %matplotlib inline

# 全局风格 / Global theme
sns.set_theme(style="whitegrid", context="notebook")

print(f"matplotlib : {plt.matplotlib.__version__}")
print(f"seaborn    : {sns.__version__}")

tips = sns.load_dataset("tips")
print(f"\nshape: {tips.shape}")
tips.head()


In [ ]:
# 看一眼数据 / Quick look
print("--- dtypes ---")
print(tips.dtypes)
print("\n--- describe (数值) ---")
print(tips.describe().round(2))
print("\n--- value_counts 类别列 ---")
for col in ["sex", "smoker", "day", "time"]:
    print(f"\n{col}:")
    print(tips[col].value_counts())


<a id="2"></a>
## 2. Matplotlib 对象模型 / The Object Model

**这是 matplotlib 最容易让人头疼的概念**——把它搞清楚，后面所有图都迎刃而解。
**The single most confusing concept** — nail it down and the rest is easy.

```
┌──────────────────────────────────────────────────┐
│ Figure（画布）                                    │
│  ┌──────────┐  ┌──────────┐                       │
│  │ Axes 1   │  │ Axes 2   │   ← 子图 / subplots    │
│  │ ┌──────┐ │  │ ┌──────┐ │                       │
│  │ │ Line │ │  │ │ Bar  │ │  ← Artists (实际画的东西) │
│  │ └──────┘ │  │ └──────┘ │                       │
│  └──────────┘  └──────────┘                       │
└──────────────────────────────────────────────────┘
```

| 概念 / Concept | 中文 | 角色 / Role |
|---|---|---|
| `Figure` | 整张画布 | 顶层容器，对应一张图片文件 |
| `Axes`   | 一个**坐标系**（不是"轴"） | 每张子图一个 Axes |
| `Axis`   | 一根坐标轴 (x 或 y) | 每个 Axes 有 2 个或 3 个 |
| `Artist` | 实际可见的元素 | 线、点、文字、图例... |

> ⚠ **`Axes`** 和 **`Axis`** 是两个不同的东西！
> `Axes` is a whole subplot; `Axis` is a single coordinate axis.


In [ ]:
# 创建 Figure 和 Axes 的标准方法 / The standard way
fig, ax = plt.subplots(figsize=(7, 4))    # 一张图，一个子图

# 在 ax 上画 / Draw on ax
x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("sin and cos")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


<a id="3"></a>
## 3. Stateful API vs OO API

Matplotlib 同一件事有**两种写法**，初学者经常乱用：
Matplotlib has **two ways** to do everything — newcomers mix them:

| API | 例子 | 何时用 / When |
|---|---|---|
| **Stateful / `pyplot`** | `plt.plot(); plt.title(); plt.xlabel()` | 快速交互探索 / quick exploration |
| **OO / `axes`** | `ax.plot(); ax.set_title(); ax.set_xlabel()` | **工业代码 / 多子图** |

> 💡 **面试 / 工业建议**：**只用 OO API**。每个图都从 `fig, ax = plt.subplots()` 开始。Stateful API 在多子图时极易混乱。
> **Interview / industry tip**: **always use OO API**. Start every plot with `fig, ax = plt.subplots()`. The stateful API gets messy with multiple subplots.


In [ ]:
# 对比同一个图的两种写法 / Same plot, two styles
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

# Left: stateful style (don't do this in real work)
plt.sca(axes[0])
plt.plot([1, 2, 3], [4, 5, 6])
plt.title("Stateful (plt.xxx)")
plt.xlabel("x")
plt.ylabel("y")

# Right: OO style — preferred
axes[1].plot([1, 2, 3], [4, 5, 6])
axes[1].set_title("OO (ax.xxx)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")

plt.tight_layout()
plt.show()


<a id="4"></a>
## 4. 基础图表 / Core Plot Types

15 种 DS 最常用的图，**每种用一句话说明什么时候用**。
The 15 most common DS plots — one-liner on when to use each.

| 图 / Plot | 何时用 / When | matplotlib API |
|---|---|---|
| **Line** 折线 | 时间序列、连续 x | `ax.plot(x, y)` |
| **Scatter** 散点 | 两个连续变量的关系 | `ax.scatter(x, y)` |
| **Bar** 柱状 | 类别 × 数值 | `ax.bar(cats, vals)` |
| **Hist** 直方图 | 一个数值变量的分布 | `ax.hist(x, bins=20)` |
| **Box** 箱线 | 类别 × 数值，含离群 | `ax.boxplot(...)` |
| **Violin** 小提琴 | 类别 × 分布形状 | `ax.violinplot(...)` |
| **Heatmap** 热图 | 矩阵/相关性 | `sns.heatmap(...)` |
| **Pair** 对图 | 多个数值变量两两 | `sns.pairplot(...)` |


In [ ]:
# 折线 + 散点 + 柱状 + 直方图 一次画完 / Four in one
fig, axes = plt.subplots(2, 2, figsize=(11, 6))

# 1. Line — sin curve
x = np.linspace(0, 2 * np.pi, 100)
axes[0, 0].plot(x, np.sin(x), color="steelblue")
axes[0, 0].set_title("Line: sin(x)")

# 2. Scatter — total_bill vs tip
axes[0, 1].scatter(tips["total_bill"], tips["tip"], alpha=0.5, s=20)
axes[0, 1].set_title("Scatter: total_bill vs tip")
axes[0, 1].set_xlabel("total_bill ($)")
axes[0, 1].set_ylabel("tip ($)")

# 3. Bar — mean tip per day
day_order = ["Thur", "Fri", "Sat", "Sun"]
day_means = tips.groupby("day", observed=True)["tip"].mean().reindex(day_order)
axes[1, 0].bar(day_means.index, day_means.values, color=["#1f77b4","#ff7f0e","#2ca02c","#d62728"])
axes[1, 0].set_title("Bar: mean tip by day")
axes[1, 0].set_ylabel("mean tip ($)")

# 4. Hist — total_bill distribution
axes[1, 1].hist(tips["total_bill"], bins=20, color="slateblue", edgecolor="white")
axes[1, 1].set_title("Histogram: total_bill")
axes[1, 1].set_xlabel("total_bill ($)")
axes[1, 1].set_ylabel("count")

plt.tight_layout()
plt.show()


**读图的标准动作 / How to read these plots**：
1. **散点图** —— 看相关性方向、是否线性、有无 outlier
2. **直方图** —— 看分布形状（对称？长尾？双峰？）+ 是否需要 log 变换
3. **柱状图** —— 比较类别均值，但**单独看均值有信息损失**（下一节用 boxplot 补充）


<a id="5"></a>
## 5. Subplot 布局 / Subplot Layouts

**多子图的三种方式**：
Three ways to do multi-panel figures:


In [ ]:
# 方法 1：plt.subplots —— 90% 的场景用它 / 90% of the time
fig, axes = plt.subplots(2, 3, figsize=(11, 5))
for i in range(2):
    for j in range(3):
        axes[i, j].plot(np.random.randn(50).cumsum())
        axes[i, j].set_title(f"axes[{i}, {j}]")
plt.tight_layout()
plt.show()


In [ ]:
# 方法 2：GridSpec —— 不规则布局 / Irregular layouts
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(10, 5))
gs = GridSpec(2, 3, figure=fig)

ax_big = fig.add_subplot(gs[0, :])           # 顶部横跨 3 列 / top span
ax_left = fig.add_subplot(gs[1, 0])
ax_mid = fig.add_subplot(gs[1, 1])
ax_right = fig.add_subplot(gs[1, 2])

ax_big.plot(np.random.randn(100).cumsum())
ax_big.set_title("top spans 3 cols")
for ax, name in zip([ax_left, ax_mid, ax_right], ["L", "M", "R"]):
    ax.hist(np.random.randn(200), bins=20)
    ax.set_title(name)

plt.tight_layout()
plt.show()


In [ ]:
# 方法 3：subplot_mosaic —— 用 ASCII 画布局，最直观 / ASCII layouts (matplotlib 3.4+)
fig, axes = plt.subplot_mosaic(
    """
    AAB
    AAC
    DEC
    """,
    figsize=(9, 5),
)
for name, ax in axes.items():
    ax.set_title(f"axes['{name}']")
    ax.plot(np.random.randn(50).cumsum())
plt.tight_layout()
plt.show()


<a id="6"></a>
## 6. 风格定制 / Styling

**面试/演示级别的图都不是默认配色**。
**Publication / interview plots are never default-styled.**


In [ ]:
# 颜色 / Color
# 1) 用名字 'red', 'blue'
# 2) HEX：'#1f77b4'
# 3) RGBA：(0.1, 0.2, 0.8, 0.6)
# 4) 颜色映射：cmap='viridis', 'plasma', 'coolwarm'

fig, ax = plt.subplots(figsize=(8, 4))

x = np.linspace(0, 10, 100)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for i, c in enumerate(colors):
    ax.plot(x, np.sin(x + i * np.pi / 4), color=c, linewidth=2.5,
            label=f"shift = {i}π/4")

ax.set_xlabel("x")
ax.set_ylabel("sin(x + φ)")
ax.set_title("Custom palette")
ax.legend(loc="lower left", frameon=True, framealpha=0.9)
ax.spines["top"].set_visible(False)         # 去掉上方边框 / hide top spine
ax.spines["right"].set_visible(False)       # 去掉右方边框 / hide right spine
plt.show()


In [ ]:
# 字号、刻度、网格 / Fonts, ticks, grid
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(tips["total_bill"], tips["tip"], "o", alpha=0.5, color="#3a86ff")
ax.set_xlabel("total bill ($)", fontsize=12)
ax.set_ylabel("tip ($)", fontsize=12)
ax.set_title("Tip vs Total Bill", fontsize=14, fontweight="bold")

# 自定义刻度 / Custom ticks
ax.set_xticks([10, 20, 30, 40, 50])
ax.tick_params(axis="both", labelsize=11)

# 仅水平网格 / Horizontal grid only
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_axisbelow(True)                  # 网格在数据之下 / grid behind data

plt.show()


> 💡 **工业小技巧 / Industry tip**：
> 1. 把 **top / right spine 隐藏**，图立刻"高级"很多。
> 2. **`alpha=0.5`** 让散点图避免黑团一坨。
> 3. 默认 `fontsize=10` 太小，把标题加到 14，标签加到 12，会议 PPT 才看得清。
> 4. 用 `set_axisbelow(True)` 把网格压到数据下面。


<a id="7"></a>
## 7. 注释 / Annotations

汇报图的灵魂——**画箭头指出关键点、写文字**。
Soul of presentation plots — arrows and text on key points.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(tips["total_bill"], tips["tip"], alpha=0.4, s=30, color="#3a86ff")

# 找最大小费的那一单 / Find the biggest-tip row
imax = tips["tip"].idxmax()
xmax, ymax = tips.loc[imax, "total_bill"], tips.loc[imax, "tip"]

# 加点醒目标记 / Highlight it
ax.scatter([xmax], [ymax], s=200, facecolor="none", edgecolor="red", linewidth=2)

# 箭头注释 / Arrow annotation
ax.annotate(
    f"max tip = ${ymax:.0f}\non a ${xmax:.0f} bill",
    xy=(xmax, ymax),                        # 箭头指向位置 / where to point
    xytext=(xmax - 20, ymax - 1),           # 文字位置 / where to place text
    arrowprops=dict(arrowstyle="->", color="red", lw=1.5),
    fontsize=11, color="darkred",
)

# 加一条平均小费的水平线 / Horizontal line at mean tip
mean_tip = tips["tip"].mean()
ax.axhline(mean_tip, color="gray", linestyle="--", linewidth=1)
ax.text(45, mean_tip + 0.1, f"mean = ${mean_tip:.2f}", color="gray")

ax.set_xlabel("total bill ($)")
ax.set_ylabel("tip ($)")
ax.set_title("Annotated scatter: tip vs total bill")
plt.show()


<a id="8"></a>
## 8. Seaborn 简介 / Seaborn Intro

Seaborn = matplotlib 上层封装，**接口直接吃 DataFrame**，专门为统计图设计。
Seaborn wraps matplotlib, **takes DataFrames directly**, designed for statistical plots.

**核心 API 三大族 / Three plot families**:
- `displot` 家族（分布）/ distribution
- `catplot` 家族（类别）/ categorical
- `relplot` 家族（关系）/ relational

每个 `xxxplot` 还有底层 `axes-level` 版本（`histplot`、`boxplot`、`scatterplot`...）。
Each `xxxplot` has a low-level **axes-level** counterpart (`histplot`, `boxplot`, `scatterplot`...).


In [ ]:
# Seaborn 的核心便利：直接传 DataFrame + 字符串列名
# Seaborn's key convenience: pass DataFrame + column-name strings
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=tips,
    x="total_bill", y="tip",
    hue="time",          # 颜色映射 / color by
    size="size",         # 点大小映射 / size by
    palette="Set2",
    alpha=0.7,
    ax=ax,
)
ax.set_title("seaborn.scatterplot with hue + size")
plt.show()


<a id="9"></a>
## 9. 分布图 / Distribution Plots

研究**一个连续变量**或**两个连续变量的联合分布**。
For one continuous variable, or the joint of two.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))

# 1) histplot —— 直方图 + 可选 KDE
sns.histplot(data=tips, x="total_bill", bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("histplot + KDE")

# 2) kdeplot —— 仅核密度估计
sns.kdeplot(data=tips, x="total_bill", hue="time", fill=True,
            common_norm=False, alpha=0.5, ax=axes[0, 1])
axes[0, 1].set_title("kdeplot by time")

# 3) ecdfplot —— 经验累计分布
# Why useful: shows ALL quantiles at once, no binning artifacts
sns.ecdfplot(data=tips, x="total_bill", hue="time", ax=axes[1, 0])
axes[1, 0].set_title("ecdfplot (cumulative distribution)")

# 4) histplot 二维 (类似 heatmap) —— 联合分布
sns.histplot(data=tips, x="total_bill", y="tip", bins=20, cbar=True, ax=axes[1, 1])
axes[1, 1].set_title("2-D histplot (joint distribution)")

plt.tight_layout()
plt.show()


> 💡 **面试常考的"分布图选择" / Common interview Q on distribution plots**：
> - 数据量小（< 100）→ **stripplot / swarmplot** 直接看点
> - 中等（100-10K）→ **histplot + KDE** 兼具结构和细节
> - 大量（> 100K）→ 仅 **histplot**（KDE 太慢）；二维用 **hexbin / 2-D hist**
> - 想比较两个分布形状 → **ecdfplot** 比 histplot 更直观，没有 bin 误差


<a id="10"></a>
## 10. 类别图 / Categorical Plots

**类别 × 数值** 的标准武器库。
The standard toolkit for **category × numeric**.

| 图 / Plot | 用途 / Use |
|---|---|
| `boxplot`    | 5 数概括 + 离群 / 5-number summary + outliers |
| `violinplot` | 完整分布形状 / full distribution shape |
| `stripplot`  | 每个点都画 / every point |
| `swarmplot`  | 避免重叠的散点 / non-overlapping points |
| `barplot`    | 均值 + CI / mean + CI |
| `countplot`  | 每类计数 / counts per category |
| `pointplot`  | 均值连线 / connected means |


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))

# 1) boxplot
sns.boxplot(data=tips, x="day", y="tip", hue="sex", ax=axes[0, 0])
axes[0, 0].set_title("boxplot")

# 2) violinplot
sns.violinplot(data=tips, x="day", y="tip", hue="sex", split=True, ax=axes[0, 1])
axes[0, 1].set_title("violinplot (split)")

# 3) stripplot
sns.stripplot(data=tips, x="day", y="tip", hue="sex", dodge=True, alpha=0.6, ax=axes[0, 2])
axes[0, 2].set_title("stripplot")

# 4) swarmplot — 不重叠点 / non-overlapping
sns.swarmplot(data=tips, x="day", y="tip", hue="sex", dodge=True, size=3, ax=axes[1, 0])
axes[1, 0].set_title("swarmplot")

# 5) barplot — 默认是均值 + 95% CI（bootstrap）
sns.barplot(data=tips, x="day", y="tip", hue="sex", ax=axes[1, 1])
axes[1, 1].set_title("barplot (mean + 95% CI)")

# 6) countplot
sns.countplot(data=tips, x="day", hue="sex", ax=axes[1, 2])
axes[1, 2].set_title("countplot")

plt.tight_layout()
plt.show()


**boxplot 解读速记 / How to read a boxplot**:

$$
\text{IQR} = Q_3 - Q_1, \quad
\text{whisker} \in [\,Q_1 - 1.5\,\text{IQR},\; Q_3 + 1.5\,\text{IQR}\,]
$$

- 盒子的上下边 = $Q_3, Q_1$（75 分位 / 25 分位）
- 盒子中间的线 = 中位数 $Q_2$
- 须的端点 = 离 box 1.5 IQR 内的最远点
- 须以外的点 = **离群值 / outliers**

> 💡 **面试坑 / Interview gotcha**：barplot 默认显示 95% bootstrap **置信区间**（不是 ±std）。误差棒**越短 ≠ 数据越集中**，是均值估计越精准。
> seaborn's barplot shows 95% bootstrap CI on the **mean**, not the data spread.


<a id="11"></a>
## 11. 关系图 / Relational Plots

**两个连续变量的关系**。
**Two continuous variables.**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# 1) scatterplot — 基本散点 + hue + size
sns.scatterplot(data=tips, x="total_bill", y="tip",
                hue="time", size="size", sizes=(20, 120),
                alpha=0.7, palette="Set2", ax=axes[0])
axes[0].set_title("scatterplot")

# 2) regplot — 散点 + 线性拟合 + 95% CI 阴影
sns.regplot(data=tips, x="total_bill", y="tip",
            scatter_kws={"alpha": 0.4}, ax=axes[1])
axes[1].set_title("regplot (linear fit + CI)")

# 3) lineplot — 默认按 x 排序后画 mean + CI
# Useful for time-series-like data; aggregates duplicate x's
day_to_int = {"Thur": 0, "Fri": 1, "Sat": 2, "Sun": 3}
tips_temp = tips.assign(day_int=tips["day"].map(day_to_int))
sns.lineplot(data=tips_temp, x="day_int", y="tip",
             hue="time", marker="o", ax=axes[2])
axes[2].set_title("lineplot (mean + CI per x)")
axes[2].set_xticks(list(day_to_int.values()))
axes[2].set_xticklabels(list(day_to_int.keys()))

plt.tight_layout()
plt.show()


In [ ]:
# lmplot —— 自带 facet 的回归图 / regression + faceting in one shot
g = sns.lmplot(
    data=tips, x="total_bill", y="tip",
    hue="smoker", col="time",
    height=4, aspect=1.1,
    scatter_kws={"alpha": 0.5},
)
g.fig.suptitle("Linear fit: tip ~ total_bill, split by smoker × time",
               y=1.02)
plt.show()


<a id="12"></a>
## 12. 矩阵图 / Heatmap & Clustermap

**面试最爱 ⭐**——很多面试官第一个问"你对这份数据做了什么 EDA"，回答里**相关性热图**是必备。
**Interview favorite ⭐** — "what EDA did you do?" → correlation heatmap is table stakes.


In [ ]:
# 相关性热图 / Correlation heatmap
numeric_cols = ["total_bill", "tip", "size"]
corr = tips[numeric_cols].corr()
print("correlation matrix:")
print(corr.round(3))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    corr, annot=True, fmt=".2f",
    cmap="coolwarm",          # 蓝—白—红 / diverging palette
    vmin=-1, vmax=1, center=0,
    square=True, cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_title("Numeric feature correlations")
plt.show()


**读相关矩阵 / Reading correlation matrices**：
- 接近 **1** → 正相关；接近 **-1** → 负相关；接近 **0** → 无线性相关
- 这里：`total_bill ↔ tip` 高度正相关 (0.68)，`size ↔ tip` 中等正相关 (0.49)
- ⚠ "无线性相关" ≠ "无关系"——非线性关系相关系数依然可以 0


In [ ]:
# 上三角遮罩：避免对称冗余 / Mask upper triangle to dedupe
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, center=0,
            square=True, ax=ax)
ax.set_title("Lower-triangle only")
plt.show()


<a id="13"></a>
## 13. FacetGrid & `pairplot`

**`pairplot`** 是 EDA 神器：一张图看遍所有数值变量两两关系 + 各自分布。
**`pairplot`** is the EDA Swiss-army knife: all pairwise scatters + per-variable distributions in one figure.


In [ ]:
# pairplot —— 多个数值变量两两关系 / All pairwise relationships
g = sns.pairplot(
    tips,
    vars=["total_bill", "tip", "size"],
    hue="time",
    diag_kind="kde",         # 对角线是 KDE / KDE on diagonal
    plot_kws={"alpha": 0.6, "s": 30},
    height=2.3,
)
g.fig.suptitle("Pairplot: tips dataset", y=1.02)
plt.show()


In [ ]:
# FacetGrid —— 通用的"分面"机制 / Generic faceting
g = sns.FacetGrid(tips, col="day", hue="sex", col_wrap=2, height=3, aspect=1.3)
g.map_dataframe(sns.scatterplot, x="total_bill", y="tip", alpha=0.6)
g.add_legend()
g.set_axis_labels("total_bill ($)", "tip ($)")
g.fig.suptitle("tip vs total_bill by day & sex", y=1.02)
plt.show()


<a id="14"></a>
## 14. 保存图片 / Saving Figures


In [ ]:
# 演示：保存 PNG / PDF 两种
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=tips, x="day", y="tip", hue="sex", ax=ax)
ax.set_title("Tip distribution by day & sex")

# PNG: 网页 / 演示 / Slack ——记得 dpi 至少 150
# PNG: web / slides — dpi ≥ 150 to look sharp
fig.savefig("/tmp/tips_box.png", dpi=200, bbox_inches="tight")

# PDF: 论文 / 报告 ——矢量，无限放大无锯齿
# PDF: paper / report — vector, infinite zoom
fig.savefig("/tmp/tips_box.pdf", bbox_inches="tight")

print("Saved:")
import os
for p in ["/tmp/tips_box.png", "/tmp/tips_box.pdf"]:
    print(f"  {p}  ({os.path.getsize(p)/1024:.1f} KB)")

plt.show()


**`bbox_inches="tight"` 几乎总要加**——否则标签/标题可能被裁掉。
**Always pass `bbox_inches="tight"`** — otherwise labels can get cropped.

**格式建议 / Format guide**：
- **PNG, dpi=200+**：屏幕展示、博客、slack
- **PDF / SVG**：论文、矢量
- **JPG**：避免（有损压缩，文字会糊）


<a id="15"></a>
## 15. 实战：Tips 完整 EDA / End-to-end EDA

把本节所有工具串起来，做一份"如果给餐厅老板看"的 EDA 报告。
Tie everything together — an EDA report for the restaurant manager.

**问题 / Questions**:
1. 什么时候小费比例最高？/ When is tip rate highest?
2. 顾客性别/吸烟状态/桌人数影响？/ Effect of sex, smoker, size?
3. 有没有可识别的高价值客户群？/ Any high-value segment?


In [ ]:
# 派生 tip_rate / Derive tip_rate
tips_eda = tips.copy()
tips_eda["tip_rate"] = tips_eda["tip"] / tips_eda["total_bill"]

print("--- tip_rate describe ---")
print(tips_eda["tip_rate"].describe().round(3))

print("\n--- by day × time ---")
print(
    tips_eda.groupby(["day", "time"], observed=True)["tip_rate"]
    .agg(["mean", "median", "count"])
    .round(3)
)


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))

# Row 1: distribution of tip rate
sns.histplot(tips_eda, x="tip_rate", bins=25, kde=True, ax=axes[0, 0])
axes[0, 0].axvline(tips_eda["tip_rate"].mean(), color="red", linestyle="--",
                   label=f"mean={tips_eda['tip_rate'].mean():.2%}")
axes[0, 0].legend()
axes[0, 0].set_title("Tip rate distribution")
axes[0, 0].set_xlabel("tip / total_bill")

sns.boxplot(data=tips_eda, x="day", y="tip_rate", hue="time", ax=axes[0, 1])
axes[0, 1].set_title("Tip rate by day × time")

sns.violinplot(data=tips_eda, x="sex", y="tip_rate", hue="smoker",
               split=True, ax=axes[0, 2])
axes[0, 2].set_title("Tip rate by sex × smoker")

# Row 2: relationships
sns.regplot(data=tips_eda, x="total_bill", y="tip",
            scatter_kws={"alpha": 0.4}, ax=axes[1, 0])
axes[1, 0].set_title("tip vs total_bill (with linear fit)")

sns.barplot(data=tips_eda, x="size", y="tip_rate", ax=axes[1, 1])
axes[1, 1].set_title("Tip rate vs party size")

corr = tips_eda[["total_bill", "tip", "size", "tip_rate"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, center=0, ax=axes[1, 2])
axes[1, 2].set_title("Correlation matrix")

plt.tight_layout()
plt.show()


### 📊 给餐厅老板的结论 / Takeaways for the manager

> ⚠ 先看样本量再下结论 —— **`Fri Lunch` 只有 7 个样本**，单元格均值看起来高（18.9%）但不可信；**`Sat Dinner` 有 87 个样本**，结论稳。
> Watch sample sizes — `Fri Lunch` (n=7) looks high but is noisy; `Sat Dinner` (n=87) is the most reliable cell.

1. **平均小费比例 ≈ 16.1%**，中位数 15.5% —— 跟美国餐饮业 15-20% 的常识一致。
   Mean tip rate ≈ **16.1%**, median 15.5%, in line with US norms.
2. **样本足够的几组里**（n > 50）：
   - 周日晚餐 **16.7%** （n=76）—— 表现最好
   - 周四午餐 **16.1%** （n=61）—— 中等
   - 周六晚餐 **15.3%** （n=87）—— 最低
   Among reliable cells, **Sunday dinners** tip best (16.7%), **Saturday dinners** worst (15.3%).
3. **吸烟者**的小费比例**方差大**（小提琴拉得很长）——有人特别大方，也有人很小气。
   Smokers' tip rates have **wider variance** — heavy tails both ways.
4. **桌人数 ↑ → 单人小费比例 ↓**（"群体扩散效应"，2-4 人组明显，5+ 人组样本太少）。
   Tip rate **decreases** with party size (free-rider effect).
5. **`total_bill ↔ tip` 强相关 (0.68)**，但 `total_bill ↔ tip_rate` 几乎为 0——**意味着小费按比例给，绝对数值由账单决定**。
   `tip` ↔ `total_bill` strong; `tip_rate` ↔ `total_bill` near zero — people tip a fixed **ratio**.

### 💡 业务建议（DS 报告的灵魂！）/ Business Recommendations

- 周末晚餐排班最优秀的服务员（小费多）
- 大桌**自动收 18% 服务费**（很多美国餐厅已经这样做）
- 不要专挑账单大的桌——`tip_rate` 不变，是绝对数值的差别
- 周五午餐潜在机会，但要先**多收数据**（n=7 太少）才能下结论


<a id="16"></a>
## 16. 小结 / Summary

| 主题 / Topic | 关键 / Key takeaway |
|---|---|
| 对象模型 | `Figure` 是画布，`Axes` 是子图；永远从 `fig, ax = plt.subplots()` 开始 |
| API 选择 | **OO API 优先**：`ax.plot()`、`ax.set_title()` |
| Subplot | `plt.subplots(r, c)` 90% 场景；`subplot_mosaic` 不规则布局 |
| 风格 | 隐藏 top/right spine、alpha=0.5、字号上调、`set_axisbelow(True)` |
| 注释 | `annotate` 加箭头 + `axhline/axvline` 标参考线 |
| Seaborn 三族 | `displot` 分布 / `catplot` 类别 / `relplot` 关系 |
| 分布 | small 用 strip；中量 hist+kde；大量 ecdf |
| 类别 | 数据少 → swarm；多 → box/violin；只要均值 → bar (含 CI) |
| 关系 | scatter / regplot / lmplot (含 facet) |
| 矩阵 | corr + heatmap = 面试必备 |
| pairplot | EDA 神器，一张看完所有两两关系 |
| 保存 | PNG dpi≥200 + `bbox_inches="tight"`；论文用 PDF |

### "选什么图"决策树 / Plot-picking decision tree

```
要展示什么？
├── 一个数值变量的分布
│   ├── < 100 点 → stripplot
│   ├── 100-100K → histplot + kde
│   └── > 100K → histplot（关掉 kde）
├── 两个数值变量的关系
│   ├── 散点 → scatterplot
│   ├── 加趋势 → regplot
│   └── 多组对比 → lmplot (col=...)
├── 类别 × 数值
│   ├── 看分布形状 → violinplot
│   ├── 看异常值 → boxplot
│   ├── 只要均值 → barplot
│   └── 数据点少要展示 → swarmplot
├── 时间序列
│   └── lineplot
└── 多变量
    ├── 相关矩阵 → heatmap
    ├── 两两关系 → pairplot
    └── 分面 → FacetGrid / lmplot(col=...)
```

### 💡 工业速查 / Industry cheat sheet

| 任务 / Task | 标准写法 / Canonical code |
|---|---|
| 单图 | `fig, ax = plt.subplots(figsize=(8,5))` |
| 多图 | `fig, axes = plt.subplots(2, 3, figsize=(15, 8))` |
| 散点高密度 | `alpha=0.3`, `s=15`, 或换 `hexbin` |
| 类别 × 数值速看 | `sns.boxplot(data=df, x=..., y=..., hue=...)` |
| 相关性 EDA | `sns.heatmap(df.corr(), annot=True, cmap="coolwarm")` |
| 时间序列 | `sns.lineplot(data=df, x="ts", y="metric", hue="segment")` |
| 演示级保存 | `fig.savefig("plot.png", dpi=200, bbox_inches="tight")` |

### 下一节预告 / Next up

**Part 0.6 · Plotly & 交互可视化** —— 把静态图升级到**可悬停 / 可缩放 / 可滚动**的交互图，做 dashboard 必备。
**Part 0.6 · Plotly & Interactive Viz** — hover, zoom, scroll. Must-have for dashboards.
